In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
import joblib

In [3]:
import wandb
wandb.login(key="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")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


True

In [4]:
import pandas as pd
import numpy as np

train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

In [5]:
rows = []

options = ['A', 'B', 'C', 'D', 'E']

for _, row in train.iterrows():

    prompt = row['prompt']
    correct = row['answer']

    for opt in options:

        text = prompt + ' [SEP] ' + str(row[opt])

        label = 1 if opt == correct else 0

        rows.append({
            'text': text,
            'label': label,
            'option': opt
        })

new_train = pd.DataFrame(rows)

print(new_train)

                                                   text  label option
0     Pick the best possible answer: What is Martin ...      0      A
1     Pick the best possible answer: What is Martin ...      1      B
2     Pick the best possible answer: What is Martin ...      0      C
3     Pick the best possible answer: What is Martin ...      0      D
4     Pick the best possible answer: What is Martin ...      0      E
...                                                 ...    ...    ...
9995  Pick the best possible answer: Which of the fo...      0      A
9996  Pick the best possible answer: Which of the fo...      0      B
9997  Pick the best possible answer: Which of the fo...      1      C
9998  Pick the best possible answer: Which of the fo...      0      D
9999  Pick the best possible answer: Which of the fo...      0      E

[10000 rows x 3 columns]


In [6]:
wandb.init(
    project="smart-mcq-solver",
    name="tfidf-logreg-v1"
)

wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260625_183253-pjniy5yp
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run tfidf-logreg-v1
wandb: ⭐️ View project at https://wandb.ai/24f2001460-iitm/smart-mcq-solver
wandb: 🚀 View run at https://wandb.ai/24f2001460-iitm/smart-mcq-solver/runs/pjniy5yp


In [7]:
from sklearn.model_selection import train_test_split

X_train, X_valid, y_train, y_valid = train_test_split(
    new_train['text'],
    new_train['label'],
    test_size=0.2,
    random_state=42
)

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1,2),
    stop_words='english'
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_valid_tfidf = vectorizer.transform(X_valid)

In [9]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(class_weight='balanced')

model.fit(X_train_tfidf, y_train)

LogisticRegression(class_weight='balanced')

In [10]:
def apk(actual, predicted, k=3):
    predicted = [predicted]

    if len(predicted) > k:
        predicted = predicted[:k]

    score = 0.0

    for i, p in enumerate(predicted):

        if p == actual:
            score = 1.0 / (i + 1)
            break

    return score


def mapk(actuals, predictions, k=3):

    return np.mean([
        apk(a, p, k)
        for a, p in zip(actuals, predictions)
    ])

In [11]:
from sklearn.metrics import accuracy_score

preds = model.predict(X_valid_tfidf)

mapk = mapk(y_valid, preds)

print(mapk)

0.9205


In [12]:
wandb.log({
    "map@3": mapk
})

In [15]:
joblib.dump(
    {
        'vectorizer':vectorizer,
        'model':model
    },
    "tfidf.pkl"
)

['tfidf.pkl']